In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
train_dir = "/content/drive/MyDrive/Dog_Cat_Classification_Week2[Om_Vaidya]/Dog_Cat_dataset/training_set"
test_dir  = "/content/drive/MyDrive/Dog_Cat_Classification_Week2[Om_Vaidya]/Dog_Cat_dataset/test_set"


In [5]:
import os

print("Train:", os.listdir(train_dir))
print("Test :", os.listdir(test_dir))


Train: ['cats', 'dogs']
Test : ['dogs', 'cats']


In [6]:
import tensorflow as tf

img_height = 150
img_width = 150
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode="binary"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode="binary"
)


Found 8015 files belonging to 2 classes.
Found 2023 files belonging to 2 classes.


In [9]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds  = test_ds.map(lambda x, y: (normalization_layer(x), y))


In [7]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


In [8]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation="relu", input_shape=(150,150,3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,735,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,828,481 (18.42 MB)

 Trainable params: 4,828,481 (18.42 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10
)


Epoch 1/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 677s 3s/step - accuracy: 0.5055 - loss: 0.7635 - val_accuracy: 0.5289 - val_loss: 0.6888
Epoch 2/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 366s 1s/step - accuracy: 0.6273 - loss: 0.6462 - val_accuracy: 0.6634 - val_loss: 0.6130
Epoch 3/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 405s 2s/step - accuracy: 0.6924 - loss: 0.5765 - val_accuracy: 0.7074 - val_loss: 0.5660
Epoch 4/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 363s 1s/step - accuracy: 0.7411 - loss: 0.5119 - val_accuracy: 0.6861 - val_loss: 0.5756
Epoch 5/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 397s 2s/step - accuracy: 0.7919 - loss: 0.4433 - val_accuracy: 0.6747 - val_loss: 0.6214
Epoch 6/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 380s 2s/step - accuracy: 0.8233 - loss: 0.3851 - val_accuracy: 0.7009 - val_loss: 0.5877
Epoch 7/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 384s 2s/step - accuracy: 0.8600 - loss: 0.3260 - val_accuracy: 0.7039 - val_loss: 0.6778
Epoch 8/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 380s 2s/step - accuracy: 0.8988 - loss: 0.2514 - val_accu

In [12]:
model.save(
    "/content/drive/MyDrive/Dog_Cat_Classification_Week2[Om_Vaidya]/dog_cat_classifier.keras"
)
